# 07 — Geometry and shortcut robustness audit

This diagnostic asks two focused questions. First, do simple image-geometry features contain label-related information in the internal training population? Second, does the saved ResNet-18 prediction change when the artificial square-padding frame or outer image border is changed in a controlled way?

This does not train a model, select a threshold, read official MURA valid, prove anatomical reasoning, or localize a lesion.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## Audit inputs

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

PROJECT_ROOT = Path(os.environ.get(
    "MURA_PROJECT_ROOT",
    "/content/drive/MyDrive/MURA_PROJECT/mura-wrist-ml-pipeline",
))
DATA_ROOT = Path(os.environ.get(
    "MURA_DATA_ROOT",
    "/content/drive/MyDrive/DATA/MURA-v1.1",
))
PREPROCESSING_DIR = Path(os.environ.get(
    "MURA_PREPROCESSING_DIR",
    "/content/drive/MyDrive/DATA/PREPROCESSED_WRIST",
))
MODEL_B_DIR = Path(os.environ.get(
    "MURA_MODEL_B_DIR",
    "/content/drive/MyDrive/DATA/MODEL_B_FINE_TUNING",
))
AUDIT_SCRIPT = PROJECT_ROOT / "scripts/run_shortcut_audit.py"
OUTPUT_DIR = Path(os.environ.get(
    "MURA_SHORTCUT_OUTPUT",
    "/content/drive/MyDrive/DATA/SHORTCUT_AUDIT",
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_MANIFEST = PREPROCESSING_DIR / "wrist_internal_train_clean.csv"
VALIDATION_MANIFEST = PREPROCESSING_DIR / "wrist_internal_val_clean.csv"
CHECKPOINT = MODEL_B_DIR / "best_resnet18_study_level.pt"
SAVED_VALIDATION_PREDICTIONS = MODEL_B_DIR / "validation_predictions.csv"
SELECTION_PATH = MODEL_B_DIR / "selection.json"

for required_path in (
    TRAIN_MANIFEST, VALIDATION_MANIFEST, CHECKPOINT,
    SAVED_VALIDATION_PREDICTIONS, SELECTION_PATH,
    DATA_ROOT / "train/XR_WRIST", AUDIT_SCRIPT,
):
    assert required_path.exists(), f"Not found: {required_path}" 

## Controlled audit

In [ ]:
# The audit checks:
# 1. Exact reproduction of saved internal-validation Model B probabilities.
# 2. A fixed geometry-only Logistic Regression with patient-grouped CV on internal train.
# 3. Changing only added square-padding pixels from black to mid-gray.
# 4. Masking the outer 5 percent border as a stress test.
# 5. Direct resize without square padding as a geometry stress test.
#
# It saves tables and aggregate charts only. It does not export MURA images.
selection = json.loads(SELECTION_PATH.read_text(encoding="utf-8"))
frozen_threshold = selection["threshold_selection"]["threshold"]

subprocess.run([
    sys.executable, str(AUDIT_SCRIPT),
    "--model-repo-root", str(PROJECT_ROOT),
    "--checkpoint", str(CHECKPOINT),
    "--data-root", str(DATA_ROOT),
    "--train-manifest", str(TRAIN_MANIFEST),
    "--validation-manifest", str(VALIDATION_MANIFEST),
    "--saved-validation-predictions", str(SAVED_VALIDATION_PREDICTIONS),
    "--frozen-threshold", str(frozen_threshold),
    "--output-dir", str(OUTPUT_DIR),
    "--batch-size", "2",
    "--workers", "0",
    "--seed", "42",
    "--device", "auto",
], check=True)

## Reproduction and geometry-only result

In [ ]:
report = json.loads((OUTPUT_DIR / "audit_report.json").read_text(encoding="utf-8"))
robustness = pd.read_csv(OUTPUT_DIR / "robustness_summary.csv")
subgroups = pd.read_csv(OUTPUT_DIR / "padding_subgroups.csv")

assert "official MURA valid/test were rejected and not read" in report["scope"]
assert report["saved_prediction_reproduction_checked"] is True
assert report["canonical_reproduction_max_abs_error"] <= 1e-5

display(pd.DataFrame([{
    "validation_studies": report["validation_studies"],
    "validation_patients": report["validation_patients"],
    "canonical_reproduction_max_abs_error": report["canonical_reproduction_max_abs_error"],
    "geometry_only_warning_AUROC": report["geometry_only_grouped_cv"]["oof_auroc"],
    "geometry_only_warning_AUPRC": report["geometry_only_grouped_cv"]["oof_auprc"],
}]))

## Perturbation summary

In [ ]:
# Do not interpret the ranking columns here as a new performance result.
# The audit is a fixed-model robustness diagnostic.
robustness_display = robustness[[
    "variant", "mean_absolute_probability_change",
    "decision_flips_at_frozen_threshold",
    "decision_flip_percent_at_frozen_threshold",
]]
display(robustness_display)
display(subgroups)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(robustness["variant"], robustness["mean_absolute_probability_change"])
axes[0].set(title="Mean absolute probability change", ylabel="Absolute change")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(robustness["variant"], robustness["decision_flip_percent_at_frozen_threshold"])
axes[1].set(title="Decision flips at the frozen threshold", ylabel="Percent of studies")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## Interpretation boundary

A geometry-only signal is a warning that acquisition geometry is associated with MURA labels in this development population. It is not proof that the ResNet uses geometry.

Changing only added padding pixels is the cleanest frame check in this notebook. Border masking and direct resize are deliberately labelled stress tests because they can also remove or distort image content. A stable result is reassuring for that intervention only; it does not prove anatomical reasoning.

Grad-CAM may be added later as a diagnostic aid. It must not be interpreted as lesion localization because MURA provides no localization ground truth.